# Olist E-Commerce Business Analytics
### E-Commerce Operations & Logistics Performance Analysis

**Role:** Data Analyst  
**Dataset:** Brazilian E-Commerce Public Dataset by Olist

## Business Problem
Evaluate fulfillment timelines, shipping charges, and customer review ratings to identify operational patterns that may affect customer satisfaction and logistics performance.

## Objectives
- Integrate order, item, product, and review data.
- Clean and validate delivered-order records.
- Measure delivery duration and delivery delays.
- Examine the relationship between delivery performance and review scores.
- Evaluate freight cost relative to product price.
- Generate actionable business insights.

> **Note:** The dataset contains order-level and item-level records. After joining order items to orders/products/reviews, the analytical table is item-level, so row counts should not be interpreted as unique order counts.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 2. Load the Core Olist CSV Files

Place the CSV files in the same folder as this notebook, or change `DATA_PATH` below.

In [ ]:
DATA_PATH = "."

orders = pd.read_csv(f"{DATA_PATH}/olist_orders_dataset.csv")
items = pd.read_csv(f"{DATA_PATH}/olist_order_items_dataset.csv")
products = pd.read_csv(f"{DATA_PATH}/olist_products_dataset.csv")
reviews = pd.read_csv(f"{DATA_PATH}/olist_order_reviews_dataset.csv")

print("Orders:", orders.shape)
print("Order items:", items.shape)
print("Products:", products.shape)
print("Reviews:", reviews.shape)

## 3. Data Integration

The project combines:
- `order_items` with `orders` using `order_id`
- the result with `products` using `product_id`
- the result with one review record per `order_id`

The review table can contain multiple review rows for an order, so duplicate order IDs are reduced before the final merge.

In [ ]:
reviews_dedup = reviews.drop_duplicates(subset=["order_id"])

df = items.merge(orders, on="order_id", how="inner")
df = df.merge(products, on="product_id", how="left")
df = df.merge(
    reviews_dedup[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

print("Master dataset shape:", df.shape)
df.to_csv("olist_master_data.csv", index=False)
df.head()

## 4. Data Cleaning

In [ ]:
df_clean = df[df["order_status"] == "delivered"].copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    df_clean[col] = pd.to_datetime(df_clean[col], errors="coerce")

df_clean = df_clean.dropna(
    subset=["order_delivered_customer_date", "review_score"]
)

df_clean["product_category_name"] = (
    df_clean["product_category_name"].fillna("unknown")
)

print("Rows after basic cleaning:", len(df_clean))

## 5. Feature Engineering

### Main analytical features
- **delivery_days:** actual days from purchase to customer delivery.
- **estimated_days:** quoted days from purchase to estimated delivery.
- **delay_days:** actual delivery date minus estimated delivery date.
- **is_delayed:** 1 when delivery occurred after the estimated date, otherwise 0.
- **freight_ratio:** freight value divided by product price (with a small denominator offset to avoid division by zero).

In [ ]:
df_clean["delivery_days"] = (
    df_clean["order_delivered_customer_date"]
    - df_clean["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

df_clean["estimated_days"] = (
    df_clean["order_estimated_delivery_date"]
    - df_clean["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 3600)

df_clean["delay_days"] = (
    df_clean["order_delivered_customer_date"]
    - df_clean["order_estimated_delivery_date"]
).dt.total_seconds() / (24 * 3600)

df_clean["is_delayed"] = (df_clean["delay_days"] > 0).astype(int)

df_clean["freight_ratio"] = df_clean["freight_value"] / (
    df_clean["price"] + 0.01
)

# Remove negative and unusually long delivery durations.
df_clean = df_clean[
    (df_clean["delivery_days"] >= 0) &
    (df_clean["delivery_days"] <= 120)
].copy()

df_clean.to_csv("olist_processed_data.csv", index=False)
print("Processed dataset shape:", df_clean.shape)

## 6. Key Performance Indicators

In [ ]:
kpis = pd.Series({
    "Processed item rows": len(df_clean),
    "Unique delivered orders": df_clean["order_id"].nunique(),
    "Average delivery days": df_clean["delivery_days"].mean(),
    "Delayed item rows (%)": df_clean["is_delayed"].mean() * 100,
    "Average review score": df_clean["review_score"].mean(),
    "On-time/early review score": df_clean.loc[df_clean["is_delayed"] == 0, "review_score"].mean(),
    "Delayed review score": df_clean.loc[df_clean["is_delayed"] == 1, "review_score"].mean(),
})
kpis

## 7. Delivery Performance vs Customer Satisfaction

A major finding is the difference in review scores between on-time/early and delayed records.

In [ ]:
delivery_review = (
    df_clean.groupby("is_delayed")["review_score"]
    .mean()
    .rename(index={0: "On-time / early", 1: "Delayed"})
)
print(delivery_review)

delivery_review.plot(kind="bar", rot=0, title="Average Review Score by Delivery Status")
plt.ylabel("Average review score")
plt.xlabel("")
plt.tight_layout()
plt.show()

## 8. Delivery Duration and Review Score

Bucket delivery durations to identify whether longer delivery times are associated with lower customer ratings.

In [ ]:
df_clean["delivery_bucket"] = pd.cut(
    df_clean["delivery_days"],
    bins=[-1, 7, 14, 21, 30, 120],
    labels=["0-7 days", "8-14 days", "15-21 days", "22-30 days", "31+ days"]
)

bucket_review = df_clean.groupby(
    "delivery_bucket", observed=False
)["review_score"].mean()

print(bucket_review)

bucket_review.plot(kind="bar", rot=0, title="Average Review Score by Delivery Duration")
plt.ylabel("Average review score")
plt.xlabel("Delivery duration")
plt.tight_layout()
plt.show()

## 9. Late Orders and One-Star Reviews

In [ ]:
late_one_star_pct = (
    (df_clean.loc[df_clean["is_delayed"] == 1, "review_score"] == 1).mean() * 100
)
print(f"Share of delayed records with a 1-star review: {late_one_star_pct:.2f}%")

## 10. Early Delivery / ETA Analysis

A negative `delay_days` means the order arrived before the quoted estimated date.

In [ ]:
early = df_clean[df_clean["delay_days"] <= 0]

early_pct = len(early) / len(df_clean) * 100
avg_early_days = -early["delay_days"].mean()

print(f"Orders/items arriving on or before ETA: {early_pct:.2f}%")
print(f"Average earliness among early records: {avg_early_days:.2f} days")

## 11. Freight Cost Analysis

`freight_ratio` measures shipping cost relative to product price. This is useful for identifying low-priced products where logistics cost can represent a large share of the purchase value.

In [ ]:
df_clean["weight_bucket"] = pd.cut(
    df_clean["product_weight_g"],
    bins=[-1, 500, 1000, 5000, np.inf],
    labels=["<500g", "500g-1kg", "1-5kg", ">5kg"]
)

freight_by_weight = (
    df_clean.groupby("weight_bucket", observed=False)["freight_ratio"]
    .mean() * 100
)

print(freight_by_weight)

freight_by_weight.plot(
    kind="bar", rot=0, title="Average Freight-to-Price Ratio by Product Weight"
)
plt.ylabel("Freight / Price (%)")
plt.xlabel("Product weight")
plt.tight_layout()
plt.show()

## 12. Product Category Analysis

The following view highlights categories with the lowest average review scores, using a minimum record threshold to reduce noise from very small groups.

In [ ]:
category_summary = (
    df_clean.groupby("product_category_name")
    .agg(
        records=("order_id", "size"),
        avg_review=("review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean")
    )
    .query("records >= 100")
    .sort_values("avg_review")
)

category_summary.head(10)

## 13. Business Insights

Based on the verified calculations from this dataset:

1. **Delivery delays are strongly associated with lower review scores.** Delayed records average about **2.55/5**, compared with about **4.21/5** for records that are not delayed.
2. About **47.02% of delayed records receive a 1-star review**, showing a substantial customer-satisfaction risk around late fulfillment.
3. The **31+ day delivery group averages about 2.23/5**, the lowest review score among the delivery-duration groups used here.
4. About **92.25% of processed records arrive on or before the estimated delivery date**. Among these early records, the average earliness is about **13.11 days**.
5. Products weighing **under 500g** show an average freight-to-price ratio of about **41.43%**, indicating that shipping can be a large component of purchase value for lightweight, lower-priced products.

These are descriptive associations, not proof that delivery time alone causes lower ratings.

## 14. Recommendations

- Monitor delayed orders as a high-priority fulfillment KPI.
- Investigate seller, carrier, destination, and product-category patterns behind late deliveries.
- Review ETA-setting practices so promised delivery dates remain useful and credible.
- Consider shipping-cost strategies for low-value/lightweight products.
- Track review score and delivery performance together rather than as separate KPIs.

## 15. Conclusion

This project demonstrates an end-to-end data analytics workflow: data integration, cleaning, feature engineering, KPI calculation, exploratory analysis, visualization, and business interpretation. The analysis shows a clear descriptive relationship between fulfillment performance and customer review outcomes and identifies freight cost as another operational consideration.

## 16. Reproducibility

Install dependencies with:

```bash
pip install -r requirements.txt
```

Then place the Olist CSV files in the notebook directory and run all notebook cells from top to bottom.

**Dataset:** Brazilian E-Commerce Public Dataset by Olist on Kaggle  
https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce